<a href="https://colab.research.google.com/github/bungei53/retail-sales-analysis-kenya/blob/main/retail_sales_analysis_2025.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

```markdown
# Retail Sales Analysis 2025: Data Cleaning

The raw sales file has many mistakes: duplicate orders, mixed date styles, spelling errors, impossible values and blanks.

```

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


```markdown
## 1. Loading the Data
I connect Google Drive and load the raw file. I keep this original copy untouched and do all the cleaning on a separate copy, so I can always go back to the starting point.
```

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv('/content/drive/MyDrive/sales_data_messy.csv')

df.head()

,order_id,order_date,customer_id,customer_age,region,city,product,category,quantity,unit_price,discount,payment_method,sales_channel,sales_rep,customer_rating,total_amount
0,ORD-4701,2025-11-27,C566,39.0,Coast,Mombasa,Smartphone,apparel,2,497.07,0.2,M-Pesa,Wholesale,David Kamau,4.0,795.31
1,ORD-2763,"Jul 18, 2025",C1043,36.0,Western,Kakamega,bookshelf,Furniture,1,119.84,0.0,Cash,Wholesale,Amina Yusuf,3.0,119.84
2,ORD-1813,2025-11-15,C917,36.0,Rift Valley,Eldoret,T-Shirt,electronics,1,$18.19,0.0,M-Pesa,Wholesale,Carol Wanjiru,5.0,18.19
3,ORD-5136,2025-02-19,C708,60.0,Nairobi,Nairobi,Desk,Furniture,1,263.01,0.1,M-Pesa,Wholesale,Carol Wanjiru,5.0,236.71
4,ORD-4763,2025-11-22,C613,36.0,Coast,Malindi,Pen Set,Stationery,5,7.62,0.0,Bank Transfer,Online,David Kamau,5.0,38.1


```markdown
## 2. Removing Duplicates
Some orders appear twice. If I leave them in, revenue and order counts would be too high.

I found 150 repeated orders and checked that each pair was identical. I removed the extra copies, leaving 5,000 unique orders.
```

In [ ]:
df.duplicated().sum()

np.int64(150)

In [ ]:
dups = df[df["order_id"].duplicated(keep=False)].sort_values("order_id")
print(dups.head(10))

      order_id    order_date customer_id  customer_age       region      city  \
2579  ORD-1080    2025-08-08        C314          41.0      Western  Kakamega   
2433  ORD-1080    2025-08-08        C314          41.0      Western  Kakamega   
2919  ORD-1201    2025-08-21        C113          21.0        COAST   Malindi   
3024  ORD-1201    2025-08-21        C113          21.0        COAST   Malindi   
3142  ORD-1220    2025-11-16        C781          43.0      Nairobi   Nairobi   
4868  ORD-1220    2025-11-16        C781          43.0      Nairobi   Nairobi   
4032  ORD-1281  Feb 18, 2025        C590          26.0        Coast   Malindi   
2205  ORD-1281  Feb 18, 2025        C590          26.0        Coast   Malindi   
4291  ORD-1326    2025-12-06        C881          25.0  Rift Valley   Eldoret   
904   ORD-1326    2025-12-06        C881          25.0  Rift Valley   Eldoret   

            product     category  quantity unit_price discount payment_method  \
2579        Pen Set   Stati

In [ ]:
before = len(df)
df_clean = df.drop_duplicates().copy()
print(f"Removed {before - len(df_clean)} duplicates. {len(df_clean)} remain.")

Removed 150 duplicates. 5000 remain.


In [ ]:
print(f'There are {df_clean.duplicated().sum()} duplicated rows remaining.')

There are 0 duplicated rows remaining.


```markdown
## 3. Fixing Numbers Stored as Text
Some prices and totals contain symbols like `$`, `%` or commas (for example `$18.19` or `1,234.50`). Because of this, the computer treated them as words instead of numbers, and I couldn't do any math with them.

I removed the symbols and converted these columns to proper numbers.
```

In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 0 to 5149
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         5000 non-null   object 
 1   order_date       5000 non-null   object 
 2   customer_id      4917 non-null   object 
 3   customer_age     4474 non-null   float64
 4   region           5000 non-null   object 
 5   city             5000 non-null   object 
 6   product          5000 non-null   object 
 7   category         5000 non-null   object 
 8   quantity         5000 non-null   int64  
 9   unit_price       4937 non-null   object 
 10  discount         4854 non-null   object 
 11  payment_method   4875 non-null   object 
 12  sales_channel    5000 non-null   object 
 13  sales_rep        4833 non-null   object 
 14  customer_rating  4543 non-null   float64
 15  total_amount     5000 non-null   object 
dtypes: float64(2), int64(1), object(13)
memory usage: 664.1+ KB


```markdown
## Fixing the Discount Scale
Some discounts were written as `15%`, which became `15` after removing the symbol, while the rest use `0.15`. I divided those by 100 so all discounts are on the same scale (0 to 0.20).
```

In [ ]:
for col in ['unit_price', 'discount', 'total_amount']:
    df_clean[col] = pd.to_numeric(
        df_clean[col].astype(str).str.replace(r'[$,%]', '', regex=True),
        errors='coerce'
    )

In [ ]:
# Fix discounts that were formatted as percentages (e.g. 15 instead of 0.15)
print("Discounts above 1:", (df_clean['discount'] > 1).sum())
df_clean.loc[df_clean['discount'] > 1, 'discount'] /= 100
print(df_clean['discount'].value_counts(dropna=False).sort_index())   # only 0, 0.05, 0.10, 0.15, 0.20 and NaN

Discounts above 1: 69
discount
0.00    2118
0.05     688
0.10     672
0.15     706
0.20     670
NaN      146
Name: count, dtype: int64


In [ ]:
df_clean.info()

<class 'pandas.core.frame.DataFrame'>
Index: 5000 entries, 0 to 5149
Data columns (total 16 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   order_id         5000 non-null   object 
 1   order_date       5000 non-null   object 
 2   customer_id      4917 non-null   object 
 3   customer_age     4474 non-null   float64
 4   region           5000 non-null   object 
 5   city             5000 non-null   object 
 6   product          5000 non-null   object 
 7   category         5000 non-null   object 
 8   quantity         5000 non-null   int64  
 9   unit_price       4937 non-null   float64
 10  discount         4854 non-null   float64
 11  payment_method   4875 non-null   object 
 12  sales_channel    5000 non-null   object 
 13  sales_rep        4833 non-null   object 
 14  customer_rating  4543 non-null   float64
 15  total_amount     5000 non-null   float64
dtypes: float64(5), int64(1), object(10)
memory usage: 664.1+ KB


```markdown
## 4. Checking for Missing and Impossible Values
First I checked how many blanks each column has. Then I looked for values that are filled in but can't be true, such as an age of 999, a rating of 10 on a 1–5 scale, or a negative quantity.

How I fixed them:
- **Impossible ages, ratings, prices and very large quantities:** turned into blanks, since I can't know the real value
- **Negative quantities:** made positive, because they look like typing mistakes with the minus sign

After this, every column is within a realistic range.
```

In [ ]:
missing = pd.DataFrame({
    'total_missing': df_clean.isnull().sum(),
    'perc_missing': (df_clean.isnull().mean() * 100).round(2)
}).sort_values('total_missing', ascending=False)

missing[missing.total_missing > 0]

,total_missing,perc_missing
customer_age,526,10.52
customer_rating,457,9.14
sales_rep,167,3.34
discount,146,2.92
payment_method,125,2.50
customer_id,83,1.66
unit_price,63,1.26


In [ ]:
print(df_clean['customer_age'].describe())
print(df_clean['customer_rating'].value_counts().sort_index())
print(df_clean['quantity'].describe())

count    4474.000000
mean       37.405454
std        42.423731
min        -5.000000
25%        27.000000
50%        35.000000
75%        43.000000
max       999.000000
Name: customer_age, dtype: float64
customer_rating
0.0        9
1.0      413
2.0      440
3.0      931
4.0     1367
5.0     1367
7.0        7
10.0       9
Name: count, dtype: int64
count    5000.000000
mean        6.329600
std        54.927767
min       -10.000000
25%         1.000000
50%         2.000000
75%         3.000000
max       999.000000
Name: quantity, dtype: float64


```markdown
## 5. Recovering Missing Values Using Math
Every order follows one simple rule: quantity × unit price × (1 − discount) = total.

If one of those pieces was missing but I knew the other pieces and the total, I calculated the missing one. This gives the exact answer instead of a guess.
```

In [ ]:
print("Age outside 18-75:   ", (~df_clean['customer_age'].between(18, 75) & df_clean['customer_age'].notna()).sum())
print("Rating outside 1-5:  ", (~df_clean['customer_rating'].between(1, 5) & df_clean['customer_rating'].notna()).sum())
print("Quantity <= 0:       ", (df_clean['quantity'] <= 0).sum())
print("Quantity > 10:       ", (df_clean['quantity'] > 10).sum())
print("Unit price <=0 or >2000:", ((df_clean['unit_price'] <= 0) | (df_clean['unit_price'] > 2000)).sum())

Age outside 18-75:    25
Rating outside 1-5:   25
Quantity <= 0:        32
Quantity > 10:        25
Unit price <=0 or >2000: 21


In [ ]:
# Age and rating: impossible values -> NaN (we can't know the real value)
df_clean.loc[~df_clean['customer_age'].between(18, 75), 'customer_age'] = np.nan
df_clean.loc[~df_clean['customer_rating'].between(1, 5), 'customer_rating'] = np.nan

# Quantity: negatives look like sign errors, so flip them; huge values are outliers
df_clean['quantity'] = df_clean['quantity'].abs()
df_clean.loc[df_clean['quantity'] > 10, 'quantity'] = np.nan

# Unit price: 0 and 99999 are impossible for these products
df_clean.loc[(df_clean['unit_price'] <= 0) | (df_clean['unit_price'] > 2000), 'unit_price'] = np.nan

In [ ]:
print(df_clean[['customer_age', 'customer_rating', 'quantity', 'unit_price']].describe())

       customer_age  customer_rating     quantity   unit_price
count   4449.000000      4518.000000  4975.000000  4916.000000
mean      35.693414         3.627490     2.680201   186.052636
std       11.299748         1.257967     2.047398   245.908372
min       18.000000         1.000000     1.000000     2.850000
25%       27.000000         3.000000     1.000000    17.220000
50%       35.000000         4.000000     2.000000    94.025000
75%       43.000000         5.000000     3.000000   249.122500
max       75.000000         5.000000    10.000000   892.330000


```markdown
## 6. Recovering Missing Quantity and Price
Every row follows `total_amount = quantity × unit_price × (1 − discount)`. When one value is missing and the other two are known, it is calculated exactly instead of guessed. This is more accurate than filling with an average.
```

In [ ]:
# Recover missing quantity where price and discount are known
mask = df_clean['quantity'].isna() & df_clean['unit_price'].notna() & df_clean['discount'].notna()
df_clean.loc[mask, 'quantity'] = (
    df_clean['total_amount'] / (df_clean['unit_price'] * (1 - df_clean['discount']))
).round()

# Recover missing unit_price where quantity and discount are known
mask = df_clean['unit_price'].isna() & df_clean['quantity'].notna() & df_clean['discount'].notna()
df_clean.loc[mask, 'unit_price'] = (
    df_clean['total_amount'] / (df_clean['quantity'] * (1 - df_clean['discount']))
).round(2)

```markdown
## 6. Making the Text Consistent
The same thing was written in different ways, like `COAST`, `coast` and `Coast`. A chart would show these as three separate groups.

I fixed this by:
- removing extra spaces and using one style of capital letters
- correcting spelling mistakes (for example, `Nairobbi` became `Nairobi`)
- using one name for each label (`Mpesa` became `M-Pesa`)
- fixing wrong categories: some products had the wrong category (like a bookshelf listed as clothing), so I assigned each product its correct category
```

In [ ]:
text_cols = ['region', 'city', 'product', 'category', 'payment_method', 'sales_channel']
for col in text_cols:
    df_clean[col] = df_clean[col].str.strip().str.title()

for col in text_cols:
    print(f"\n{col}:\n", df_clean[col].value_counts(dropna=False))


region:
 region
Coast          1264
Rift Valley    1256
Nairobi        1254
Western        1226
Name: count, dtype: int64

city:
 city
Malindi     663
Thika       650
Eldoret     635
Nakuru      621
Kisumu      602
Kakamega    600
Nairobi     588
Mombasa     578
Kisumo       24
Mombassa     23
Nairobbi     16
Name: count, dtype: int64

product:
 product
T-Shirt          442
Notebook         442
Headphones       438
Smartphone       432
Laptop           422
Running Shoes    420
Bookshelf        415
Desk             412
Office Chair     405
Jacket           393
Pen Set          390
Printer Paper    389
Name: count, dtype: int64

category:
 category
Electronics    1269
Apparel        1235
Furniture      1232
Stationery     1221
Electronic       43
Name: count, dtype: int64

payment_method:
 payment_method
Bank Transfer    1279
M-Pesa           1225
Cash             1155
Credit Card      1153
NaN               125
Mpesa              63
Name: count, dtype: int64

sales_channel:
 sales_chan

In [ ]:
# 1. Typos in city
df_clean['city'] = df_clean['city'].replace({
    'Kisumo': 'Kisumu',
    'Mombassa': 'Mombasa',
    'Nairobbi': 'Nairobi'
})

# 2. Payment method and sales channel labels
df_clean['payment_method'] = df_clean['payment_method'].replace({'Mpesa': 'M-Pesa'})
df_clean['sales_channel'] = df_clean['sales_channel'].replace({'Retail': 'Retail Store'})

# 3. Rebuild category from product
product_to_category = {
    'Laptop': 'Electronics', 'Smartphone': 'Electronics', 'Headphones': 'Electronics',
    'Office Chair': 'Furniture', 'Desk': 'Furniture', 'Bookshelf': 'Furniture',
    'Notebook': 'Stationery', 'Pen Set': 'Stationery', 'Printer Paper': 'Stationery',
    'Running Shoes': 'Apparel', 'Jacket': 'Apparel', 'T-Shirt': 'Apparel'
}
df_clean['category'] = df_clean['product'].map(product_to_category)

In [ ]:
pd.crosstab(df_clean['product'], df_clean['category'])

category,Apparel,Electronics,Furniture,Stationery
product,,,,
Bookshelf,0,0,415,0
Desk,0,0,412,0
Headphones,0,438,0,0
Jacket,393,0,0,0
Laptop,0,422,0,0
Notebook,0,0,0,442
Office Chair,0,0,405,0
Pen Set,0,0,0,390
Printer Paper,0,0,0,389


In [ ]:
for col in ['city', 'payment_method', 'sales_channel', 'category']:
    print(f"\n{col}:\n", df_clean[col].value_counts(dropna=False))

print(pd.crosstab(df_clean['product'], df_clean['category']))


city:
 city
Malindi     663
Thika       650
Eldoret     635
Kisumu      626
Nakuru      621
Nairobi     604
Mombasa     601
Kakamega    600
Name: count, dtype: int64

payment_method:
 payment_method
M-Pesa           1288
Bank Transfer    1279
Cash             1155
Credit Card      1153
NaN               125
Name: count, dtype: int64

sales_channel:
 sales_channel
Online          1741
Retail Store    1667
Wholesale       1592
Name: count, dtype: int64

category:
 category
Electronics    1292
Apparel        1255
Furniture      1232
Stationery     1221
Name: count, dtype: int64
category       Apparel  Electronics  Furniture  Stationery
product                                                   
Bookshelf            0            0        415           0
Desk                 0            0        412           0
Headphones           0          438          0           0
Jacket             393            0          0           0
Laptop               0          422          0           0
Note

```markdown
## 7. Fixing the Dates
Dates were written in three different styles, like `2025-03-14`, `14/03/2025` and `Mar 14, 2025`.

I told the computer exactly how to read each style instead of letting it guess. Guessing is risky, because it can swap the day and month without warning (for example, reading 10 March as 3 October).

A few dates are impossible, like 31 February. I left them blank instead of inventing a date, since I can't know the real one.
```

In [ ]:
# Try each format separately; rows that don't match that format become NaT
d1 = pd.to_datetime(df_clean['order_date'], format='%Y-%m-%d', errors='coerce')
d2 = pd.to_datetime(df_clean['order_date'], format='%d/%m/%Y', errors='coerce')
d3 = pd.to_datetime(df_clean['order_date'], format='%b %d, %Y', errors='coerce')

# Combine: use d1 where it worked, otherwise d2, otherwise d3
df_clean['order_date'] = d1.fillna(d2).fillna(d3)

print("Unparsed dates:", df_clean['order_date'].isna().sum())
print(df_clean['order_date'].min(), "to", df_clean['order_date'].max())
print(df_clean['order_date'].dtype)

Unparsed dates: 21
2025-01-01 00:00:00 to 2025-12-31 00:00:00
datetime64[ns]


In [ ]:
# Creating Time-based helper columns
df_clean['month'] = df_clean['order_date'].dt.month
df_clean['month_name'] = df_clean['order_date'].dt.strftime('%b')
df_clean['quarter'] = df_clean['order_date'].dt.quarter

```markdown
## 8. Filling the Remaining Blanks
I handled each column differently, depending on whether a good answer is possible:
- **Discount:** worked out from the total, since the math shows which discount was used
- **Price:** filled with the typical price of that same product
- **Sales rep, payment method, customer ID:** marked as `Unknown`, because I can't guess these
- **Age:** filled with the typical age, and I added a column showing which ones were filled in
- **Customer rating:** left blank on purpose. Filling in made-up ratings would give a false picture of customer satisfaction
- **Dates:** the impossible ones stay blank
```

In [ ]:
# 1. Discount: derive from the total, then snap to the nearest valid discount
valid_discounts = np.array([0, 0.05, 0.10, 0.15, 0.20])
m = df_clean['discount'].isna() & df_clean[['quantity', 'unit_price']].notna().all(axis=1)
implied = 1 - df_clean['total_amount'] / (df_clean['quantity'] * df_clean['unit_price'])
# Map and fill
if m.sum() > 0:
    df_clean.loc[m, 'discount'] = [valid_discounts[np.abs(valid_discounts - x).argmin()] for x in implied[m]]

# 2. Unit price: median price of the same product
df_clean['unit_price'] = df_clean['unit_price'].fillna(
    df_clean.groupby('product')['unit_price'].transform('median')
)

# 3. Labels that can't be known
for col in ['sales_rep', 'payment_method', 'customer_id']:
    df_clean[col] = df_clean[col].fillna('Unknown')

# 4. Age: flag, then fill with the median
df_clean['age_was_missing'] = df_clean['customer_age'].isna()
df_clean['customer_age'] = df_clean['customer_age'].fillna(df_clean['customer_age'].median())

# 5. The last few discounts (price or quantity was also missing): use the most common value
df_clean['discount'] = df_clean['discount'].fillna(0)

# customer_rating, order_date and 1 quantity are left missing on purpose

```markdown
## 10. Missing Values Post-Cleaning Check
Let's view the missing values distribution after running all the strategic treatments.
```

In [ ]:
missing_post = pd.DataFrame({
    'total_missing': df_clean.isnull().sum(),
    'perc_missing': (df_clean.isnull().mean() * 100).round(2)
}).sort_values('total_missing', ascending=False)

display(missing_post[missing_post.total_missing > 0])

,total_missing,perc_missing
customer_rating,482,9.64
order_date,21,0.42
month,21,0.42
quarter,21,0.42
month_name,21,0.42
quantity,1,0.02


```markdown
## 9. Final Check and Saving
Before saving, the notebook automatically checks that the data follows the rules: no duplicates, realistic ages and ratings, quantities between 1 and 10, and one category per product. The clean file is then saved for the next stage (charts and reporting).

**Summary:** 150 duplicates removed, text and number formats fixed, impossible values corrected, categories repaired, and dates standardized. The final dataset has 5,000 clean orders.
```

In [ ]:
assert df_clean.duplicated().sum() == 0
assert df_clean['customer_age'].between(18, 75).all()
assert df_clean['customer_rating'].dropna().between(1, 5).all()
assert df_clean['quantity'].dropna().between(1, 10).all()
assert df_clean['discount'].between(0, 0.20).all()
assert df_clean.groupby('product')['category'].nunique().max() == 1
print("All checks passed")

print("Cleaned shape:", df_clean.shape)    # should be (5000, 20)
df_clean.to_csv('/content/drive/MyDrive/sales_data_clean.csv', index=False)

All checks passed
Cleaned shape: (5000, 20)


```markdown
## 12. Summary of Cleaning Actions
- **Duplicates Removed:** 150 rows successfully removed.
- **Discounts Rescaled:** Re-scaled values that were raw percentages above 1.0 (such as 15.0 to 0.15).
- **Invalid Values Remedied:** Invalid ages (outside 18-75), ratings (outside 1-5), and quantities / prices set to null or corrected.
- **Standardized Text:** Cities, sales channels, payment methods, and product categories reconstructed and cleaned.
- **Unparsed Dates:** Handled explicitly; invalid dates kept as missing safely.
```